In [ ]:
import json
import pandas as pd
from transformers import pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score
)

# =====================================================
# 1. CHARGEMENT DU CORPUS (MultiNLI)
# =====================================================

DATA_PATH = "../data/raw/multinli_1.0/multinli_1.0_train.jsonl"

data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["gold_label"] != "-":
            data.append(item)

data = data[:100]

df = pd.DataFrame(data)[["sentence1", "sentence2", "gold_label"]]

print("Distribution des labels :")
print(df["gold_label"].value_counts(normalize=True))
print("-" * 50)

In [ ]:
# =====================================================
# 2. SPLIT TRAIN / DEV / TEST
# =====================================================

train_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df["gold_label"]
)

train_df, dev_df = train_test_split(
    train_df, test_size=0.176, random_state=42, stratify=train_df["gold_label"]
)

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")
print("-" * 50)

In [ ]:
# =====================================================
# 3. CHARGEMENT DU MODELE NLI
# =====================================================

nli = pipeline(
    task="text-classification",
    model="roberta-large-mnli",
    return_all_scores=True
)

In [ ]:
# =====================================================
# 4. PREDICTION AVEC SEUILS
# =====================================================

ENTAILMENT_THRESHOLD = 0.8
CONTRADICTION_THRESHOLD = 0.8

def predict_nli(sentence1, sentence2):
    output = nli({"text": sentence1, "text_pair": sentence2})
    # output is a list of dicts; iterate directly
    scores = {x['label'].lower(): float(x['score']) for x in output}

    if scores["entailment"] >= ENTAILMENT_THRESHOLD:
        decision = "VRAI"
        label = "entailment"
        confidence = scores["entailment"]

    elif scores["contradiction"] >= CONTRADICTION_THRESHOLD:
        decision = "FAUX"
        label = "contradiction"
        confidence = scores["contradiction"]

    else:
        decision = "A_VERIFIER"
        label = "neutral"
        confidence = scores["neutral"]

    return label, decision, confidence

In [ ]:
# =====================================================
# 5. EVALUATION SUR LE TEST
# =====================================================

predicted_labels = []
decisions = []
confidences = []

for _, row in test_df.iterrows():
    label, decision, conf = predict_nli(row["sentence1"], row["sentence2"])
    predicted_labels.append(label)
    decisions.append(decision)
    confidences.append(conf)

test_df = test_df.copy()
test_df["predicted_label"] = predicted_labels
test_df["decision"] = decisions
test_df["confidence"] = confidences

In [ ]:
# =====================================================
# 6. METRIQUES
# =====================================================

accuracy = accuracy_score(test_df["gold_label"], test_df["predicted_label"])
kappa = cohen_kappa_score(test_df["gold_label"], test_df["predicted_label"])
conf_matrix = confusion_matrix(test_df["gold_label"], test_df["predicted_label"])

print("Accuracy :", round(accuracy * 100, 2), "%")
print("Cohen Kappa :", round(kappa, 3))
print("\nMatrice de confusion :")
print(conf_matrix)

print("\nRapport de classification :")
print(classification_report(
    test_df["gold_label"],
    test_df["predicted_label"]
))

In [ ]:
# =====================================================
# 7. ANALYSE SIMPLE DES ERREURS
# =====================================================

errors = test_df[test_df["gold_label"] != test_df["predicted_label"]]
print(f"Nombre d'erreurs : {len(errors)} / {len(test_df)}")

print("\nExemples d'erreurs :")
print(errors[[
    "sentence1",
    "sentence2",
    "gold_label",
    "predicted_label",
    "confidence"
]].head(5))

In [ ]:
sentence1 = "Yanis is smart"
sentence2 = "Yanis is stupid"

label, decision, confidence = predict_nli(sentence1, sentence2)
print(f"Phrase 1 : {sentence1}")
print(f"Phrase 2 : {sentence2}")
print(f"Décision : {decision} (label: {label}, confiance: {round(confidence, 3)})")
